# 11 · LSTM (recorrente)

Rede recorrente (Hochreiter & Schmidhuber, 1997).

Desenho adotado:
- LSTM de 2 camadas, 128 unidades, dropout 0.1 entre as camadas, lendo a janela passo a passo;
- previsão **direta**: o último estado oculto passa por `Linear(128, H)` e o modelo prevê os 96 passos de uma vez. A alternativa autorregressiva acumula erro e é muito mais lenta.

Não existe configuração publicada de LSTM para estas bases. Os valores acima são escolhas usuais e ficam fixos. Só o learning rate é ajustado, igual aos outros modelos.

In [1]:
# Configuração: igual em todos os notebooks
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJETO = '/content/drive/MyDrive/RN-Projeto'
except ImportError:          # fora do Colab (ex.: rodando no computador)
    PROJETO = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJETO, 'src'))
print('Pasta do projeto:', PROJETO)

import importlib
import rn_utils as ru
importlib.reload(ru)        # pega a versão mais nova se o 02 foi rodado de novo
ru.configurar(PROJETO)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/RN-Projeto
Projeto: /content/drive/MyDrive/RN-Projeto | dispositivo: cpu


## Modelo

`criar(L, H)` devolve um `nn.Module` que recebe `[lote, L, 1]` e devolve `[lote, H, 1]`. A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos os modelos.

In [2]:
import torch
import torch.nn as nn

class LSTMPrev(nn.Module):
    def __init__(self, L, H, oculto=128, camadas=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=oculto, num_layers=camadas,
                            batch_first=True, dropout=dropout)
        self.saida = nn.Linear(oculto, H)

    def forward(self, x):                      # [B, L, 1] → [B, H, 1]
        sequencia, _ = self.lstm(x)            # [B, L, oculto]
        return self.saida(sequencia[:, -1])[..., None]

def criar(L, H):
    return LSTMPrev(L, H)

In [3]:
# Verificação do formato de entrada e saída
import torch
for L in (96, 720):
    m = criar(L, ru.HORIZONTE)
    saida = m(torch.randn(4, L, 1))
    assert saida.shape == (4, ru.HORIZONTE, 1), saida.shape
    print(f'L={L}: saída {tuple(saida.shape)} ✓ | {sum(p.numel() for p in m.parameters()):,} parâmetros')

L=96: saída (4, 96, 1) ✓ | 211,552 parâmetros
L=720: saída (4, 96, 1) ✓ | 211,552 parâmetros


## Treino e avaliação

O learning rate é escolhido pela validação (L = 336, seed 0) entre {1e-3, 3e-4, 1e-4}. Depois o modelo é treinado em todas as janelas e seeds das duas bases e avaliado no teste, com a janela original e com as quatro janelas embaralhadas. As execuções já salvas em `resultados/` são apenas recarregadas e listadas.

In [4]:
ru.rodar_grade('LSTM', criar)

lr escolhido para LSTM em ETTh1: 0.0001  (val: {'0.001': 0.0924, '0.0003': 0.0928, '0.0001': 0.0904})
  · ETTh1_LSTM_L96_H96_lr0.0001_s0 (salvo) | val 0.1055 | teste MSE 0.0577 MAE 0.1834 | embaralhar_tudo ×1.21 | 12 épocas, 11s (cuda)
  · ETTh1_LSTM_L96_H96_lr0.0001_s1 (salvo) | val 0.1079 | teste MSE 0.0584 MAE 0.1830 | embaralhar_tudo ×1.18 | 11 épocas, 9s (cuda)
  · ETTh1_LSTM_L96_H96_lr0.0001_s2 (salvo) | val 0.1064 | teste MSE 0.0593 MAE 0.1887 | embaralhar_tudo ×1.35 | 16 épocas, 13s (cuda)
  · ETTh1_LSTM_L192_H96_lr0.0001_s0 (salvo) | val 0.0919 | teste MSE 0.0571 MAE 0.1827 | embaralhar_tudo ×1.38 | 16 épocas, 24s (cuda)
  · ETTh1_LSTM_L192_H96_lr0.0001_s1 (salvo) | val 0.0921 | teste MSE 0.0594 MAE 0.1871 | embaralhar_tudo ×1.31 | 16 épocas, 24s (cuda)
  · ETTh1_LSTM_L192_H96_lr0.0001_s2 (salvo) | val 0.0930 | teste MSE 0.0567 MAE 0.1812 | embaralhar_tudo ×1.35 | 14 épocas, 21s (cuda)
  · ETTh1_LSTM_L336_H96_lr0.0001_s0 (salvo) | val 0.0904 | teste MSE 0.0531 MAE 0.1778 | emb

## Resumo (média das 3 seeds)

`×tudo`, `×antigo`, `×recente` e `×blocos` = MSE com a janela embaralhada / MSE original. ×1,00 significa que o modelo ignora aquela ordem.

In [5]:
df = ru.carregar_resultados()
r = df[df.modelo == 'LSTM']
resumo = r.groupby(['dataset', 'L']).agg(**{
    'MSE val': ('mse_val', 'mean'), 'MSE teste': ('original_mse', 'mean'), 'dp': ('original_mse', 'std'),
    'MAE teste': ('original_mae', 'mean'), 'MSE sem suspeitos': ('original_mse_ss', 'mean'),
    **{'×' + p.replace('embaralhar_', ''): (f'{p}_razao', 'mean') for p in ru.PERTURBACOES},
    'épocas': ('epocas_rodadas', 'mean'), 'tempo (s)': ('tempo_treino_s', 'mean')})
resumo.round(4)

MSE val  MSE teste      dp  MAE teste  MSE sem suspeitos   ×tudo  \
dataset L                                                                       
ETTh1   96    0.1066     0.0585  0.0008     0.1851             0.0575  1.2494   
        192   0.0924     0.0577  0.0014     0.1837             0.0592  1.3461   
        336   0.0913     0.0546  0.0014     0.1798             0.0562  1.4719   
        512   0.0972     0.0518  0.0024     0.1747             0.0533  1.4037   
        720   0.0990     0.0497  0.0005     0.1721             0.0501  1.5001   
Weather 96    0.5426     0.5094  0.0115     0.4900             0.5094  1.3305   
        336   0.5194     0.5563  0.0314     0.5121             0.5563  1.2466   
        720   0.5888     0.4909  0.0260     0.4925             0.4909  1.8161   

             ×antigo  ×recente  ×blocos   épocas  tempo (s)  
dataset L                                                    
ETTh1   96    1.0075    1.0726   1.2821  13.0000    11.2599  
        192   1.0033    1.1601   1.4676  15.3333    22.6863  
        336   1.0642    1.2164   1.6193  18.0000    43.9702  
        512   0.9901    1.2757   1.6436  16.0000    58.0967  
        720   0.9979    1.3687   1.7057  19.0000    94.8643  
Weather 96    1.0801    1.0964   1.4057  12.6667    44.2336  
        336   1.0581    1.0533   1.5268  11.6667   122.1704  
        720   1.1179    1.2700   2.0364  10.3333   224.5693